In [8]:
from pyspark.sql import SparkSession

In [9]:
spark = SparkSession.builder \
    .appName("Flight_Data_HDFS_SQL") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

In [ ]:
'''
path_hdfs = "hdfs://localhost:9000/data/finalterm/"
df_flights = spark.read.csv(path_hdfs + "flights_cleaned.csv", header=True, inferSchema=True)
df_airlines = spark.read.csv(path_hdfs + "airlines_cleaned.csv", header=True, inferSchema=True)
df_airports = spark.read.csv(path_hdfs + "airports_cleaned.csv", header=True, inferSchema=True)
'''

In [5]:
'''
df_flights.createOrReplaceTempView("flights")
df_airlines.createOrReplaceTempView("airlines")
df_airports.createOrReplaceTempView("airports")
'''

# QUERY 1: Systemic Failure Detection – Airports with the Worst On-Time Departure Rate

In [ ]:
# QUERY 1: Systemic Failure Detection – Airports with the Worst On-Time Departure Rate
query1 = spark.sql("""
                   SELECT f.ORIGIN_AIRPORT                  AS airport_code,
                          a.CITY                            AS city,
                          a.STATE                           AS state,
                          COUNT(*)                          AS total_flights,
                          ROUND(SUM(CASE WHEN f.DEPARTURE_DELAY > 15 THEN 1 ELSE 0 END) * 100.0
                                    / COUNT(*), 2)          AS delay_rate_pct,
                          ROUND(AVG(f.DEPARTURE_DELAY), 2)  AS avg_dep_delay_min,
                          ROUND(AVG(f.AIR_SYSTEM_DELAY), 2) AS avg_air_system_delay,
                          ROUND(AVG(f.AIRLINE_DELAY), 2)    AS avg_airline_delay,
                          ROUND(AVG(f.WEATHER_DELAY), 2)    AS avg_weather_delay
                   FROM flights f
                            LEFT JOIN airports a ON f.ORIGIN_AIRPORT = a.IATA_CODE
                   WHERE f.CANCELLED = 0
                     AND f.DEPARTURE_DELAY IS NOT NULL
                   GROUP BY f.ORIGIN_AIRPORT, a.CITY, a.STATE
                   HAVING COUNT(*) >= 1000
                   ORDER BY delay_rate_pct DESC LIMIT 20
                   """)

print("=== Q1: Airports with Worst On-Time Departure Rate (by Delay Source) ===")
query1.show(truncate=False)

In [ ]:
# QUERY 2: Geographical Weather Bottlenecks – States Most Affected by Weather Delay
query2 = spark.sql("""
                   SELECT a.STATE                                                               AS state,
                          COUNT(*)                                                              AS total_departures,
                          ROUND(SUM(CASE WHEN f.WEATHER_DELAY > 0 THEN 1 ELSE 0 END) * 100.0
                                    / COUNT(*), 2)                                              AS weather_affected_pct,
                          ROUND(AVG(CASE WHEN f.WEATHER_DELAY > 0 THEN f.WEATHER_DELAY END), 2) AS avg_weather_delay_when_affected,
                          SUM(CASE
                                  WHEN f.CANCELLED = 1
                                      AND f.CANCELLATION_REASON = 'B' THEN 1
                                  ELSE 0 END)                                                   AS weather_cancellations,
                          ROUND(SUM(CASE
                                        WHEN f.CANCELLED = 1
                                            AND f.CANCELLATION_REASON = 'B' THEN 1
                                        ELSE 0 END) * 100.0
                                    / COUNT(*),
                                2)                                                              AS weather_cancel_rate_pct
                   FROM flights f
                            LEFT JOIN airports a ON f.ORIGIN_AIRPORT = a.IATA_CODE
                   WHERE a.STATE IS NOT NULL
                   GROUP BY a.STATE
                   HAVING COUNT(*) >= 500
                   ORDER BY weather_affected_pct DESC LIMIT 20
                   """)
print("=== Q2: Geographical Weather Bottlenecks by State ===")
query2.show(truncate=False)

# QUERY 3: High-Frequency, High-Cancellation Flight Paths


In [ ]:
query3 = spark.sql("""
                   SELECT f.ORIGIN_AIRPORT                                             AS origin,
                          ap1.CITY                                                     AS origin_city,
                          f.DESTINATION_AIRPORT                                        AS destination,
                          ap2.CITY                                                     AS dest_city,
                          COUNT(*)                                                     AS total_flights,
                          SUM(f.CANCELLED)                                             AS total_cancellations,
                          ROUND(SUM(f.CANCELLED) * 100.0 / COUNT(*), 2)                AS cancellation_rate_pct,
                          SUM(CASE WHEN f.CANCELLATION_REASON = 'A' THEN 1 ELSE 0 END) AS cancelled_by_airline,
                          SUM(CASE WHEN f.CANCELLATION_REASON = 'B' THEN 1 ELSE 0 END) AS cancelled_by_weather,
                          SUM(CASE WHEN f.CANCELLATION_REASON = 'C' THEN 1 ELSE 0 END) AS cancelled_by_nas,
                          SUM(CASE WHEN f.CANCELLATION_REASON = 'D' THEN 1 ELSE 0 END) AS cancelled_by_security
                   FROM flights f
                            LEFT JOIN airports ap1 ON f.ORIGIN_AIRPORT = ap1.IATA_CODE
                            LEFT JOIN airports ap2 ON f.DESTINATION_AIRPORT = ap2.IATA_CODE
                   GROUP BY f.ORIGIN_AIRPORT, ap1.CITY, f.DESTINATION_AIRPORT, ap2.CITY
                   HAVING COUNT(*) >= 500                            -- chỉ xét route tần suất cao
                      AND SUM(f.CANCELLED) * 100.0 / COUNT(*) >= 2.0 -- tỷ lệ hủy >= 2%
                   ORDER BY cancellation_rate_pct DESC, total_flights DESC LIMIT 20
                   """)
print("=== Q3: High-Frequency, High-Cancellation Flight Paths ===")
query3.show(truncate=False)

# QUERY 4: Cascading Delay Analysis – Identifying Airlines with Late Aircraft Problem

In [ ]:
query4 = spark.sql("""
                   SELECT al.AIRLINE                           AS airline_name,
                          COUNT(*)                             AS total_delayed_flights,
                          ROUND(AVG(f.DEPARTURE_DELAY), 2)     AS avg_total_dep_delay,
                          ROUND(AVG(f.LATE_AIRCRAFT_DELAY), 2) AS avg_late_aircraft_delay,
                          ROUND(AVG(f.AIRLINE_DELAY), 2)       AS avg_airline_delay,
                          ROUND(AVG(f.AIR_SYSTEM_DELAY), 2)    AS avg_air_system_delay,
                          ROUND(AVG(f.WEATHER_DELAY), 2)       AS avg_weather_delay,
                          ROUND(
                                  AVG(f.LATE_AIRCRAFT_DELAY) * 100.0 /
                                  NULLIF(AVG(f.AIR_SYSTEM_DELAY) + AVG(f.AIRLINE_DELAY) +
                                         AVG(f.LATE_AIRCRAFT_DELAY) + AVG(f.WEATHER_DELAY) +
                                         AVG(f.SECURITY_DELAY), 0),
                                  2)                           AS late_aircraft_share_pct
                   FROM flights f
                            JOIN airlines al ON f.AIRLINE = al.IATA_CODE
                   WHERE f.CANCELLED = 0
                     AND f.DEPARTURE_DELAY > 15
                   GROUP BY al.AIRLINE
                   ORDER BY late_aircraft_share_pct DESC
                   """)

print("=== Q4: Cascading Delay Analysis (Late Aircraft Share per Airline) ===")
query4.show(truncate=False)

# QUERY 5: Monthly Delay Seasonality – Detecting Systemic Seasonal Bottlenecks

In [ ]:
query5 = spark.sql("""
                   SELECT
                       MONTH, COUNT (*) AS total_flights, ROUND(SUM (CANCELLED) * 100.0 / COUNT (*), 2) AS cancellation_rate_pct, ROUND(AVG (CASE WHEN CANCELLED = 0 THEN DEPARTURE_DELAY END), 2) AS avg_dep_delay, ROUND(AVG (CASE WHEN CANCELLED = 0 THEN WEATHER_DELAY END), 2) AS avg_weather_delay, ROUND(AVG (CASE WHEN CANCELLED = 0 THEN AIR_SYSTEM_DELAY END), 2) AS avg_air_system_delay, ROUND(AVG (CASE WHEN CANCELLED = 0 THEN AIRLINE_DELAY END), 2) AS avg_airline_delay, ROUND(AVG (CASE WHEN CANCELLED = 0 THEN LATE_AIRCRAFT_DELAY END), 2) AS avg_late_aircraft_delay
                   FROM flights
                   GROUP BY MONTH
                   ORDER BY MONTH
                   """)
print("=== Q5: Monthly Delay Seasonality – Systemic Seasonal Bottlenecks ===")
query5.show(truncate=False)


Q6 — Hiệu ứng "Đổ vỡ Domino" (Late Aircraft Delay Cascading Effect)

In [7]:
query_q6 = """
WITH flight_sequence AS (
    SELECT 
        f.AIRLINE,
        al.AIRLINE as AIRLINE_NAME,
        f.ORIGIN_AIRPORT,
        f.DATE,
        f.SCHEDULED_DEPARTURE,
        f.label,
        LAG(f.LATE_AIRCRAFT_DELAY, 1, 0) OVER (
            PARTITION BY f.AIRLINE, f.ORIGIN_AIRPORT 
            ORDER BY f.DATE, f.SCHEDULED_DEPARTURE
        ) as prev_late_aircraft_delay
    FROM flights f
    LEFT JOIN airlines al ON f.AIRLINE = al.IATA_CODE
    WHERE f.CANCELLED = 0
)
SELECT 
    AIRLINE_NAME,
    COUNT(*) as total_domino_flights,
    ROUND(AVG(label) * 100, 2) || '%' as domino_delay_rate
FROM flight_sequence
WHERE prev_late_aircraft_delay > 0 
GROUP BY AIRLINE_NAME
ORDER BY AVG(label) DESC
"""
spark.sql(query_q6).show(truncate=False)

+----------------------------+--------------------+-----------------+
|AIRLINE_NAME                |total_domino_flights|domino_delay_rate|
+----------------------------+--------------------+-----------------+
|JetBlue Airways             |31670               |41.34%           |
|American Eagle Airlines Inc.|32101               |41.07%           |
|Spirit Air Lines            |11715               |40.69%           |
|Frontier Airlines Inc.      |11632               |39.37%           |
|Southwest Airlines Co.      |163166              |37.52%           |
|Virgin America              |5394                |36.24%           |
|United Air Lines Inc.       |49851               |35.77%           |
|Atlantic Southeast Airlines |54756               |35.29%           |
|Delta Air Lines Inc.        |50104               |33.72%           |
|American Airlines Inc.      |57576               |33.36%           |
|US Airways Inc.             |13642               |32.5%            |
|Skywest Airlines In

Q7 - Hãng bay nào có hành động cắt giảm chuyến bay mạnh tay nhất trên các tuyến bay "bất ổn"

In [ ]:
query_q7 = """
WITH monthly_route_volume AS (
    SELECT 
        f.AIRLINE,
        al.AIRLINE as AIRLINE_NAME,
        f.ORIGIN_AIRPORT || ' -> ' || f.DESTINATION_AIRPORT as route,
        f.MONTH,
        COUNT(*) as current_month_flights,
        ROUND(AVG(f.label) * 100, 2) as monthly_delay_rate
    FROM flights f
    LEFT JOIN airlines al ON f.AIRLINE = al.IATA_CODE
    GROUP BY f.AIRLINE, al.AIRLINE, f.ORIGIN_AIRPORT, f.DESTINATION_AIRPORT, f.MONTH
),
route_trends AS (
    SELECT 
        AIRLINE_NAME,
        route,
        MONTH,
        current_month_flights,
        monthly_delay_rate,
        LAG(current_month_flights, 1) OVER (
            PARTITION BY AIRLINE_NAME, route 
            ORDER BY MONTH
        ) as prev_month_flights
    FROM monthly_route_volume
)
SELECT 
    AIRLINE_NAME,
    route,
    MONTH as drop_month,
    prev_month_flights as flights_before,
    current_month_flights as flights_after,
    (current_month_flights - prev_month_flights) as volume_change,
    monthly_delay_rate || '%' as delay_rate_at_drop_time
FROM route_trends
WHERE prev_month_flights IS NOT NULL 
  AND current_month_flights < (prev_month_flights * 0.5)
ORDER BY volume_change ASC
LIMIT 15
"""
spark.sql(query_q7_new).show(15, truncate=False)

+---------------------------+----------+----------+--------------+-------------+-------------+-----------------------+
|AIRLINE_NAME               |route     |drop_month|flights_before|flights_after|volume_change|delay_rate_at_drop_time|
+---------------------------+----------+----------+--------------+-------------+-------------+-----------------------+
|Skywest Airlines Inc.      |LAX -> SAN|11        |425           |177          |-248         |12.43%                 |
|Skywest Airlines Inc.      |SAN -> LAX|11        |424           |176          |-248         |11.93%                 |
|Skywest Airlines Inc.      |SMF -> LAX|11        |327           |100          |-227         |6.0%                   |
|Skywest Airlines Inc.      |LAX -> SMF|11        |323           |99           |-224         |10.1%                  |
|Delta Air Lines Inc.       |BOS -> LGA|7         |383           |184          |-199         |27.17%                 |
|Delta Air Lines Inc.       |LGA -> BOS|7       

Q8 — Các "Siêu Sân bay Trung chuyển" rủi ro nhất nước Mỹ (High-Risk Hub Airports Analysis)

In [ ]:
query_q8 = """
WITH airport_volumes AS (
    SELECT 
        ORIGIN_AIRPORT,
        COUNT(*) as total_departures,
        PERCENT_RANK() OVER (ORDER BY COUNT(*)) as volume_rank
    FROM flights
    GROUP BY ORIGIN_AIRPORT
),
mega_hubs AS (
    SELECT ORIGIN_AIRPORT 
    FROM airport_volumes 
    WHERE volume_rank >= 0.90
)
SELECT 
    f.ORIGIN_AIRPORT as airport_code,
    COALESCE(ap.AIRPORT, 'Unknown (Numeric Code Airport)') as airport_full_name,
    COUNT(*) as total_flights,
    ROUND(AVG(f.CANCELLED) * 100, 2) || '%' as cancellation_rate,
    ROUND(AVG(f.label) * 100, 2) || '%' as delay_rate
FROM flights f
INNER JOIN mega_hubs mh ON f.ORIGIN_AIRPORT = mh.ORIGIN_AIRPORT
LEFT JOIN airports ap ON f.ORIGIN_AIRPORT = ap.IATA_CODE
GROUP BY f.ORIGIN_AIRPORT, ap.AIRPORT
ORDER BY AVG(f.label) DESC
LIMIT 10
"""
spark.sql(query_q8).show(truncate=False)

+------------+----------------------------------------------------------------------+-------------+-----------------+----------+
|airport_code|airport_full_name                                                     |total_flights|cancellation_rate|delay_rate|
+------------+----------------------------------------------------------------------+-------------+-----------------+----------+
|ORD         |Chicago O'Hare International Airport                                  |285884       |2.99%            |22.62%    |
|LGA         |LaGuardia Airport (Marine Air Terminal)                               |99605        |4.55%            |22.13%    |
|MIA         |Miami International Airport                                           |69341        |1.13%            |22.04%    |
|DEN         |Denver International Airport                                          |196055       |1.08%            |21.32%    |
|BWI         |Baltimore-Washington International Airport                            |86079       

Q9 — Chỉ số chống chịu thời tiết (Weather Resilience Index)

In [ ]:
query_q9 = """
SELECT 
    al.AIRLINE as airline_name,
    COUNT(*) as flights_caught_in_storm,
    ROUND(AVG(f.WEATHER_DELAY), 2) as avg_weather_delay_minutes,
    ROUND(1 - AVG(f.label), 4) as weather_resilience_index 
FROM flights f
LEFT JOIN airlines al ON f.AIRLINE = al.IATA_CODE
WHERE f.CANCELLED = 0 AND f.WEATHER_DELAY > 0
GROUP BY al.AIRLINE
ORDER BY weather_resilience_index DESC
"""
spark.sql(query_q9).show(truncate=False)

+----------------------------+-----------------------+-------------------------+------------------------+
|airline_name                |flights_caught_in_storm|avg_weather_delay_minutes|weather_resilience_index|
+----------------------------+-----------------------+-------------------------+------------------------+
|Virgin America              |2072                   |17.02                    |0.0541                  |
|Hawaiian Airlines Inc.      |572                    |19.98                    |0.0332                  |
|Alaska Airlines Inc.        |867                    |44.79                    |0.0208                  |
|Spirit Air Lines            |925                    |47.66                    |0.0162                  |
|United Air Lines Inc.       |7531                   |43.16                    |0.0157                  |
|Delta Air Lines Inc.        |11838                  |50.93                    |0.0147                  |
|Skywest Airlines Inc.       |4426            

Q10 - Top 1 khung giờ cao điểm có lượng chuyến bay cất cánh lớn nhất của từng Bang

In [ ]:
query_q10 = """
WITH hourly_volumes AS (
    SELECT 
        COALESCE(ap.STATE, 'Unknown State') as origin_state,
        SUBSTRING(f.DEPARTURE_TIME, 1, 2) as departure_hour,
        COUNT(*) as flight_count
    FROM flights f
    LEFT JOIN airports ap ON f.ORIGIN_AIRPORT = ap.IATA_CODE
    WHERE f.DEPARTURE_TIME != '-1'
    GROUP BY COALESCE(ap.STATE, 'Unknown State'), SUBSTRING(f.DEPARTURE_TIME, 1, 2)
),
ranked_hours AS (
    SELECT 
        origin_state,
        departure_hour,
        flight_count,
        DENSE_RANK() OVER (PARTITION BY origin_state ORDER BY flight_count DESC) as rnk
    FROM hourly_volumes
)
SELECT 
    origin_state,
    departure_hour,
    flight_count,
    rnk as rank
FROM ranked_hours
WHERE rnk = 1
ORDER BY origin_state, rnk
"""
spark.sql(query_q10).show(40, truncate=False)

+------------+--------------+------------+----+
|origin_state|departure_hour|flight_count|rank|
+------------+--------------+------------+----+
|AK          |13            |2393        |1   |
|AL          |06            |3265        |1   |
|AR          |06            |2388        |1   |
|AS          |23            |99          |1   |
|AZ          |10            |13431       |1   |
|CA          |06            |49259       |1   |
|CO          |11            |20164       |1   |
|CT          |06            |2147        |1   |
|DE          |21            |22          |1   |
|FL          |07            |30042       |1   |
|GA          |19            |27729       |1   |
|GU          |06            |180         |1   |
|HI          |13            |8237        |1   |
|IA          |06            |1976        |1   |
|ID          |06            |2437        |1   |
|IL          |13            |28461       |1   |
|IN          |06            |4741        |1   |
|KS          |06            |1319       